In [ ]:
import pandas as pd
import psycopg2
import json
from psycopg2 import extras
import html

In [ ]:
dsn = "dbname=BeautifulSoup_not_cleand user=postgres password=kenan123 host=localhost port=5432"
q = """SELECT * FROM raw_products"""
with psycopg2.connect(dsn) as con:
        with con.cursor() as cur:
            cur.execute(q)
            result = cur.fetchall()
        con.commit()


In [ ]:
df = pd.DataFrame(result , columns=['id' , 'title' , 'description' , 'price' , 'size' , 'color' , 'sku' , 'category' , 'photo_url' , 'extracted_at'])

In [ ]:
df['price'] = df['price'].astype(float)

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated(subset=['sku']).sum()

In [ ]:
df['title'] = df['title'].str.strip()

In [ ]:
df['size'] = df['size'].replace('NA' , 'no sizes')

In [ ]:
df['color'] = df['color'].replace('NA' , 'no colors')

In [ ]:
df['category'] = df['category'].str.lower().str.replace('category:', '')

In [ ]:

df['description'] = df['description'].apply(html.unescape)

In [ ]:
mask = (df['price'] < 0).sum()
print(mask)

In [ ]:
q = """
    INSERT INTO clean_raw_products (id, title, description, price, size, color, sku, category, photo_url,extracted_at)
    VALUES ( %s, %s, %s, %s, %s, %s, %s, %s,%s,%s)
    ON CONFLICT (id) DO NOTHING;
    """

data = [tuple(x) for x in df.values]
with psycopg2.connect(dsn) as con:
        with con.cursor() as cur:
            extras.execute_batch(cur,q,data)
        con.commit()